In [1]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import dgl
from dgl.data import AmazonCoBuyComputerDataset

def seed_all(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)

seed_all(42)

DATA = Path("../data/amazon_computers.pt")
OUT = Path("../outputs/")
OUT.mkdir(parents=True, exist_ok=True)


In [2]:
def sample_per_class(rs, onehot, n, forbidden=None):
    n_samples, n_classes = onehot.shape
    per = {c: [] for c in range(n_classes)}
    for c in range(n_classes):
        for i in range(n_samples):
            if onehot[i, c] and (forbidden is None or i not in set(forbidden)):
                per[c].append(i)
    return np.concatenate([rs.choice(per[c], n, replace=False) for c in per])

def make_splits(rs, labels):
    n_classes = int(labels.max() + 1)
    N = len(labels)
    oh = np.eye(n_classes)[labels.numpy()]
    tr = sample_per_class(rs, oh, 20)
    va = sample_per_class(rs, oh, 30, forbidden=tr)
    te = np.setdiff1d(np.arange(N), np.concatenate([tr, va]))
    return torch.LongTensor(tr), torch.LongTensor(va), torch.LongTensor(te)


In [3]:
if DATA.exists():
    ck = torch.load(DATA, weights_only=False)
    g, feats, labels = ck["graph"], ck["feats"], ck["labels"]
    idx_tr, idx_va, idx_te = ck["idx_train"], ck["idx_val"], ck["idx_test"]
else:
    ds = AmazonCoBuyComputerDataset(raw_dir=str(Path.home() / ".dgl"))
    g = dgl.add_self_loop(ds[0])
    feats = g.ndata["feat"]
    labels = g.ndata["label"]
    rs = np.random.RandomState(42)
    idx_tr, idx_va, idx_te = make_splits(rs, labels)
    torch.save({
        "graph": g, "feats": feats, "labels": labels,
        "idx_train": idx_tr, "idx_val": idx_va, "idx_test": idx_te
    }, DATA)

print(f"N={g.num_nodes()} E={g.num_edges()} feat={feats.shape} C={int(labels.max()+1)}")
print(f"train {len(idx_tr)} val {len(idx_va)} test {len(idx_te)}")

src, dst = g.edges()
lab = labels.numpy()
mask = src.numpy() != dst.numpy()
hom = (lab[src.numpy()[mask]] == lab[dst.numpy()[mask]]).mean()
print(f"homophily {hom:.3f}  avg_degree {(g.num_edges()-g.num_nodes())/g.num_nodes():.1f}")


N=13752 E=505474 feat=torch.Size([13752, 767]) C=10
train 200 val 300 test 13252
homophily 0.777  avg_degree 35.8
